# Proyecto RappiPlus: de datos a decisiones de negocio

**Introducción**


El objetivo de este proyecto es evaluar el desempeño del servicio **RappiPlus** para apoyar **decisiones de negocio basadas en datos**.

Se trabajan con múltiples datasets del negocio:

- **rappiplus_orders_raw.csv** → información de pedidos, precios, descuentos y revenue  
- **rappiplus_catalog.csv** → costos de productos, categorías y proveedores  
- **rappiplus_marketing_spend.csv** → inversión en marketing por canal y país  
- **events / users / user_activity (SQL)** → comportamiento del usuario dentro de la plataforma  
- **experiment_checkout_ui.csv** → resultados de un experimento A/B en el checkout  

El análisis sigue una lógica clara y progresiva:

1. 🔍 Evaluar si podemos confiar en los datos (calidad de datos en Python) 

2. 💰 Analizar si el negocio es rentable (revenue, costos y profit)  

3. 🛒 Entender dónde se pierden los usuarios (funnel de conversión)  

4. 🔁 Evaluar si los usuarios regresan (retención por cohortes)  

5. 🧪 Validar si los cambios generan impacto (test estadístico)  

6. 📊 Comunicar los resultados (dashboard en BI)  

A lo largo del proyecto, se transforman datos en insights para responder preguntas clave del negocio y proponer **recomendaciones accionables**.

---

## 🔹 Paso 1: Cargar y validar la calidad de los datos

---

### 1.1 Carga de datos y vista rápida

**🎯 Objetivo:** Familiarizarte con la estructura de los datasets del negocio antes de analizarlos.

**Instrucciones:**

- Importa las librerías necesarias
- Carga los archivos:
  - `rappiplus_orders_raw.csv`
  - `rappiplus_catalog.csv`
  - `rappiplus_marketing_spend.csv`
- Guarda los DataFrames en:
  - `orders`, `catalog`, `marketing`
- Explora cada dataset.

---

In [1]:
# importar librerías
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

In [2]:
# cargar archivos
orders = pd.read_csv('https://practicum-content.s3.amazonaws.com/datasets/rappiplus_orders_raw.csv')
catalog = pd.read_csv('https://practicum-content.s3.amazonaws.com/datasets/rappiplus_catalog.csv')
marketing = pd.read_csv('https://practicum-content.s3.amazonaws.com/datasets/rappiplus_marketing_spend.csv')

In [3]:
# explorar datasets
display(orders.head())

,id_pedido,id_usuario,fecha_hora_pedido,pais,dispositivo,fuente_referencia,nombre_producto,categoria_producto,cantidad,precio_unitario,monto_descuento,monto_total
0,order_0,user_6993,2025-05-22,Argentina,desktop,organic,Jacket-Winter-M,Moda,2.0,332.69,0.0,665.37
1,order_1,user_1329,2025-06-15,Mexico,desktop,paid_search,Tablet-Standard-64GB,Electronica,1.0,176.86,5.0,171.86
2,order_2,user_3194,2025-05-02,Argentina,desktop,social,Blender-XL-Red,Hogar,2.0,102.99,10.0,195.99
3,order_3,user_4510,2025-06-09,Colombia,mobile,social,Tablet-Standard-64GB,Electronica,1.0,257.87,15.0,242.87
4,order_4,user_5044,2025-03-30,Argentina,desktop,paid_search,Blender-XL-Red,Hogar,1.0,336.28,0.0,336.28


In [4]:
display(catalog.head())

,nombre_producto,categoria_producto,costo_unitario,proveedor
0,Laptop-Gaming-16GB,Electrónica,280.68,"Fuller, Pena and Myers"
1,Phone-Pro-128GB,Electrónica,10.12,King Ltd
2,Tablet-Standard-64GB,Electrónica,25.21,Bowers LLC
3,Blender-XL-Red,Hogar,176.64,Long-Reid
4,Vacuum-Pro-Black,Hogar,16.60,"Rivera, Carr and Finley"


In [5]:
display(marketing.head())

,fecha,pais,id_campaña,canal,gasto
0,2025-01-01,Mexico,organic_Mexico,organic,2446.25
1,2025-01-01,Mexico,paid_search_Mexico,paid_search,2704.34
2,2025-01-01,Mexico,social_Mexico,social,2045.01
3,2025-01-01,Colombia,organic_Colombia,organic,2597.21
4,2025-01-01,Colombia,paid_search_Colombia,paid_search,1771.40


In [6]:
# 1. Create a dictionary mapping the table name to the DataFrame variable
tables = {
    "Orders": orders,
    "Catalog": catalog,
    "Marketing": marketing
}

# 2. Loop through the dictionary and run .info() on each
for table_name, df in tables.items():
    print(f"--- Resumen de la tabla: {table_name} ---")
    df.info()
    print("\n" + "="*50 + "\n") # Adds a visual separator between tables

--- Resumen de la tabla: Orders ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 25100 entries, 0 to 25099
Data columns (total 12 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   id_pedido           25100 non-null  object 
 1   id_usuario          25100 non-null  object 
 2   fecha_hora_pedido   25100 non-null  object 
 3   pais                24800 non-null  object 
 4   dispositivo         25080 non-null  object 
 5   fuente_referencia   25070 non-null  object 
 6   nombre_producto     25070 non-null  object 
 7   categoria_producto  25020 non-null  object 
 8   cantidad            25050 non-null  float64
 9   precio_unitario     25050 non-null  float64
 10  monto_descuento     25050 non-null  float64
 11  monto_total         25100 non-null  float64
dtypes: float64(4), object(8)
memory usage: 2.3+ MB


--- Resumen de la tabla: Catalog ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7 entries, 0 to 6
Data colum

- Tabla de **orders**

In [7]:
cols_to_analyze = ['cantidad', 'precio_unitario', 'monto_descuento', 'monto_total']

for col in cols_to_analyze:
    print(f"\n--- Resumen de {col} ---")
    display(orders[col].describe())


--- Resumen de cantidad ---


count    25050.000000
mean         7.092735
std        296.277003
min         -2.000000
25%          1.000000
50%          2.000000
75%          2.000000
max      20000.000000
Name: cantidad, dtype: float64


--- Resumen de precio_unitario ---


count    25050.000000
mean       259.305549
std        138.726461
min         20.030000
25%        138.377500
50%        258.715000
75%        380.332500
max        499.960000
Name: precio_unitario, dtype: float64


--- Resumen de monto_descuento ---


count    25050.000000
mean         4.500798
std          5.223010
min          0.000000
25%          0.000000
50%          0.000000
75%         10.000000
max         15.000000
Name: monto_descuento, dtype: float64


--- Resumen de monto_total ---


count    2.510000e+04
mean     2.072680e+03
std      9.894995e+04
min     -4.926500e+02
25%      1.805075e+02
50%      3.417500e+02
75%      5.185800e+02
max      8.840200e+06
Name: monto_total, dtype: float64

- Tabla de **catalog**

In [8]:
cols_to_analyze = ['costo_unitario']

for col in cols_to_analyze:
    print(f"\n--- Resumen de {col} ---")
    display(catalog[col].describe())


--- Resumen de costo_unitario ---


count      7.000000
mean     102.252857
std      111.011563
min       10.120000
25%       16.905000
50%       25.210000
75%      182.975000
max      280.680000
Name: costo_unitario, dtype: float64

- Tabla de **marketing**

In [9]:
cols_to_analyze = ['gasto']

for col in cols_to_analyze:
    print(f"\n--- Resumen de {col} ---")
    display(marketing[col].describe())


--- Resumen de gasto ---


count    1620.00000
mean     1772.74292
std       734.43294
min       501.11000
25%      1128.03000
50%      1782.42500
75%      2420.68500
max      2999.36000
Name: gasto, dtype: float64

---

### Revisión y calidad de datos

**🎯 Objetivo:** Detectar y corregir problemas en los datos que puedan afectar el análisis de revenue, costos y rentabilidad.

Se revisan los 3 datasets
- Validar y convertir fechas al formato correcto  
- Revisar variables numéricas (sin negativos o ceros inválidos)  
- Verificar consistencia de montos  
- Eliminar duplicados  
- Revisar variables categóricas 

---

In [10]:
nulos_orders = orders.isna().sum()
nulos_catalog = catalog.isna().sum()
nulos_marketing = marketing.isna().sum()

nulos_tables = {
    "orders": nulos_orders,
    "catalog": nulos_catalog,
    "marketing": nulos_marketing
}

for table_name, nul_col in nulos_tables.items():
    print(f"Cantidad de nulos en la tabla {table_name} \n{nul_col}\n")

Cantidad de nulos en la tabla orders 
id_pedido               0
id_usuario              0
fecha_hora_pedido       0
pais                  300
dispositivo            20
fuente_referencia      30
nombre_producto        30
categoria_producto     80
cantidad               50
precio_unitario        50
monto_descuento        50
monto_total             0
dtype: int64

Cantidad de nulos en la tabla catalog 
nombre_producto       0
categoria_producto    0
costo_unitario        0
proveedor             0
dtype: int64

Cantidad de nulos en la tabla marketing 
fecha           0
pais            0
id_campaña      0
canal         101
gasto           0
dtype: int64



In [11]:
# Verificar duplicados en la tabla 'orders'

keys_orders = ['id_pedido']

print(f"La cantidad de duplicados en la tabla 'orders' es: {orders.duplicated(subset=keys_orders).sum()}")
pct_dup = (orders.duplicated(subset=keys_orders).sum() / (~orders['id_pedido'].isna()).sum()) * 100
print(f"El porcentaje de duplicados en la columna 'id_pedido' de la tabla 'orders' es: {pct_dup:.4f}")
print()

display(orders[orders.duplicated(subset=keys_orders, keep=False)].sort_values(keys_orders).head())

La cantidad de duplicados en la tabla 'orders' es: 100
El porcentaje de duplicados en la columna 'id_pedido' de la tabla 'orders' es: 0.3984



,id_pedido,id_usuario,fecha_hora_pedido,pais,dispositivo,fuente_referencia,nombre_producto,categoria_producto,cantidad,precio_unitario,monto_descuento,monto_total
25023,order_10082,user_690,2025-02-17,Argentina,desktop,social,Sneakers-Urban-42,Moda,2.0,221.34,0.0,442.67
10082,order_10082,user_690,2025-02-17,Argentina,desktop,social,Sneakers-Urban-42,Moda,2.0,221.34,0.0,442.67
25037,order_10709,user_6783,2025-02-16,Colombia,mobile,organic,Jacket-Winter-M,Moda,1.0,170.10,10.0,160.10
10709,order_10709,user_6783,2025-02-16,Colombia,mobile,organic,Jacket-Winter-M,Moda,1.0,170.10,10.0,160.10
25065,order_10829,user_7697,2025-01-23,Argentina,mobile,social,Phone-Pro-128GB,Electronica,2.0,115.54,0.0,231.08


In [12]:
# Verificar duplicatos en la tabla 'marketing'
keys_marketing = ['id_campaña']
print(f"La cantidad de duplicados en la tabla 'marketing' es: {marketing.duplicated(subset=keys_marketing).sum()}")

marketing[marketing.duplicated(subset=keys_marketing, keep=False)].sort_values(keys_marketing).head()

La cantidad de duplicados en la tabla 'marketing' es: 1611


,fecha,pais,id_campaña,canal,gasto
789,2025-03-29,Argentina,organic_Argentina,organic,2417.71
1185,2025-05-12,Argentina,organic_Argentina,organic,1437.93
1248,2025-05-19,Argentina,organic_Argentina,organic,1508.45
1482,2025-06-14,Argentina,organic_Argentina,organic,2427.72
1383,2025-06-03,Argentina,organic_Argentina,organic,2925.69


In [13]:
# Eliminar duplicados de la tabla 'orders'
orders_clean = orders.copy()

orders_clean = orders_clean.drop_duplicates(subset=keys_orders, keep='first').reset_index(drop=True)
print(f"La cantidad de duplicados en la tabla 'orders' es: {orders_clean.duplicated(subset=keys_orders).sum()}")

La cantidad de duplicados en la tabla 'orders' es: 0


In [14]:
# la columna 'fecha' en la tabla marketing debe ser convertida a formato date
# La columna 'fecha_hora_pedido' en la tabla orders debe ser convertida a formato date
orders_clean['fecha_hora_pedido'] = pd.to_datetime(orders['fecha_hora_pedido'], errors='coerce')
print(f"El tipo de datos de la columna 'fecha_hora_pedido' de la tabla 'orders' es: {orders_clean['fecha_hora_pedido'].dtype}, antes era {orders['fecha_hora_pedido'].dtype}")

marketing_clean = marketing.copy()
marketing_clean['fecha'] = pd.to_datetime(marketing['fecha'], errors='coerce')
print(f"El tipo de datos de la columna 'fecha' de la tabla 'marketing' es: {marketing_clean['fecha'].dtype}, antes era {marketing['fecha'].dtype}")


El tipo de datos de la columna 'fecha_hora_pedido' de la tabla 'orders' es: datetime64[ns], antes era object
El tipo de datos de la columna 'fecha' de la tabla 'marketing' es: datetime64[ns], antes era object


In [15]:
# Bucle for para saber los valores unicos de las columnas categorias de la tabla de 'orders'
cols_num = ['cantidad', 'precio_unitario', 'monto_descuento']
cols_cat = ['pais', 'dispositivo', 'fuente_referencia', 'nombre_producto', 'categoria_producto']

for col in cols_cat:
    print(f"Los valores unicos en la columna {col} son {orders_clean[col].unique()}")

Los valores unicos en la columna pais son ['Argentina' 'Mexico' 'Colombia' 'mexico' 'colombia' 'argentina' nan]
Los valores unicos en la columna dispositivo son ['desktop' 'mobile' nan]
Los valores unicos en la columna fuente_referencia son ['organic' 'paid_search' 'social' nan]
Los valores unicos en la columna nombre_producto son ['Jacket-Winter-M' 'Tablet-Standard-64GB' 'Blender-XL-Red'
 'Laptop-Gaming-16GB' 'Sneakers-Urban-42' 'Phone-Pro-128GB'
 'Vacuum-Pro-Black' nan]
Los valores unicos en la columna categoria_producto son ['Moda' 'Electronica' 'Hogar' nan]


In [16]:
# Filtro para saber si el precio unitario es consistente para el producto 'Sneakers-Urban-42'
orders_clean[(orders_clean['nombre_producto'] == 'Sneakers-Urban-42') & (orders_clean['fuente_referencia'] == 'social') & (orders_clean['pais'] == 'Mexico') & (orders_clean['dispositivo'] == 'mobile')].sort_values(by='precio_unitario', ascending=False).head()

,id_pedido,id_usuario,fecha_hora_pedido,pais,dispositivo,fuente_referencia,nombre_producto,categoria_producto,cantidad,precio_unitario,monto_descuento,monto_total
6172,order_6172,user_4601,2025-03-10,Mexico,mobile,social,Sneakers-Urban-42,Moda,2.0,499.18,5.0,993.36
1830,order_1830,user_5964,2025-04-15,Mexico,mobile,social,Sneakers-Urban-42,Moda,1.0,497.24,15.0,482.24
13821,order_13821,user_4319,2025-02-19,Mexico,mobile,social,Sneakers-Urban-42,Moda,1.0,497.19,10.0,487.19
17839,order_17839,user_6284,2025-05-05,Mexico,mobile,social,Sneakers-Urban-42,Moda,1.0,496.70,0.0,496.70
2210,order_2210,user_2088,2025-04-06,Mexico,mobile,social,Sneakers-Urban-42,Moda,2.0,493.92,5.0,982.85


In [17]:
# Filtro para saber si el precio unitario es consistente para el producto 'Phone-Pro-128GB'
orders_clean[(orders_clean['nombre_producto'] == 'Phone-Pro-128GB') & (orders_clean['fuente_referencia'] == 'social') & (orders_clean['pais'] == 'Mexico') & (orders_clean['dispositivo'] == 'mobile')].sort_values(by='precio_unitario', ascending=False).head()

,id_pedido,id_usuario,fecha_hora_pedido,pais,dispositivo,fuente_referencia,nombre_producto,categoria_producto,cantidad,precio_unitario,monto_descuento,monto_total
24483,order_24483,user_5912,2025-03-18,Mexico,mobile,social,Phone-Pro-128GB,Electronica,1.0,499.40,0.0,499.40
17688,order_17688,user_7841,2025-04-20,Mexico,mobile,social,Phone-Pro-128GB,Electronica,2.0,490.36,0.0,980.72
10143,order_10143,user_6421,2025-02-02,Mexico,mobile,social,Phone-Pro-128GB,Electronica,2.0,484.71,15.0,954.43
12850,order_12850,user_7472,2025-01-07,Mexico,mobile,social,Phone-Pro-128GB,Electronica,2.0,480.14,0.0,960.29
10700,order_10700,user_4984,2025-02-07,Mexico,mobile,social,Phone-Pro-128GB,Electronica,2.0,474.30,0.0,948.59


In [18]:
# Verificar los valores nulos de la columna 'categoria_producto'
orders_clean[orders_clean['categoria_producto'].isna()].head()

,id_pedido,id_usuario,fecha_hora_pedido,pais,dispositivo,fuente_referencia,nombre_producto,categoria_producto,cantidad,precio_unitario,monto_descuento,monto_total
44,order_44,user_2899,2025-02-22,Colombia,desktop,NaN,NaN,NaN,1.0,318.38,5.0,313.38
45,order_45,user_6196,2025-02-10,Mexico,mobile,NaN,NaN,NaN,2.0,354.06,10.0,698.12
46,order_46,user_5815,2025-04-24,Mexico,mobile,NaN,NaN,NaN,1.0,359.31,10.0,349.31
47,order_47,user_406,2025-05-08,Colombia,mobile,NaN,NaN,NaN,2.0,476.78,0.0,953.56
48,order_48,user_6187,2025-01-26,Colombia,desktop,NaN,NaN,NaN,2.0,137.96,0.0,275.93


In [19]:
# Verificar si se puede saber el pais de los campos nulos a traves del id_usuario
orders_clean[orders_clean['id_usuario'] == 'user_2444']

,id_pedido,id_usuario,fecha_hora_pedido,pais,dispositivo,fuente_referencia,nombre_producto,categoria_producto,cantidad,precio_unitario,monto_descuento,monto_total
128,order_128,user_2444,2025-05-06,NaN,desktop,social,Vacuum-Pro-Black,Hogar,2.0,218.43,0.0,436.86
1413,order_1413,user_2444,2025-04-12,mexico,desktop,organic,Blender-XL-Red,Hogar,1.0,133.85,0.0,133.85
11334,order_11334,user_2444,2025-03-09,Mexico,desktop,social,Phone-Pro-128GB,Electronica,2.0,155.04,0.0,310.08
12450,order_12450,user_2444,2025-06-25,Mexico,desktop,organic,Tablet-Standard-64GB,Electronica,1.0,263.37,15.0,248.37
22422,order_22422,user_2444,2025-02-05,Mexico,desktop,social,Blender-XL-Red,Hogar,2.0,132.49,5.0,259.98
23377,order_23377,user_2444,2025-05-16,Mexico,desktop,social,Blender-XL-Red,Hogar,2.0,224.40,0.0,448.80
24258,order_24258,user_2444,2025-01-17,Mexico,desktop,organic,Sneakers-Urban-42,Moda,2.0,391.42,0.0,782.83


In [20]:
# TABLA 'orders'
# Como manejar los nulos en estas columnas categoricas ['pais', 'dispositivo', 'fuente_referencia', 'nombre_producto', 'categoria_producto'].
# Como manejar los nulos en estas columnas numericas ['cantidad', 'precio_unitario', 'monto_descuento']
# Imputar con mean, median, etc.

# Estandarizar la columna 'pais'
orders_clean['pais'] = orders_clean['pais'].str.strip().str.title()

# Estandarizar la columna 'dispositivo'
orders_clean['dispositivo'] = orders_clean['dispositivo'].str.strip().str.title()

# Estandarizar la columna 'fuente_referencia'
orders_clean['fuente_referencia'] = orders_clean['fuente_referencia'].str.strip().str.title()

# Fill NaN values using other rows from the same user
orders_clean['pais'] = orders_clean.groupby('id_usuario')['pais'].transform(lambda x: x.ffill().bfill())

# Convertir la columna numerica 'monto_descuento' de float64 a int64
orders_clean['monto_descuento'] = orders_clean['monto_descuento'].fillna(0).astype('int')
print(f"El tipo de dato de la columna 'monto_descuento' ahora es {orders_clean['monto_descuento'].dtype}")


El tipo de dato de la columna 'monto_descuento' ahora es int64


In [21]:
orders_clean.nunique().sort_values()

dispositivo               2
pais                      3
fuente_referencia         3
categoria_producto        3
monto_descuento           4
cantidad                  6
nombre_producto           7
fecha_hora_pedido       181
id_usuario             7642
precio_unitario       19543
monto_total           21536
id_pedido             25000
dtype: int64

In [22]:
orders_clean.isna().mean().sort_values(ascending=False) * 100

categoria_producto    0.320
cantidad              0.200
precio_unitario       0.200
fuente_referencia     0.120
nombre_producto       0.120
dispositivo           0.080
pais                  0.076
id_pedido             0.000
id_usuario            0.000
fecha_hora_pedido     0.000
monto_descuento       0.000
monto_total           0.000
dtype: float64

In [23]:
# Eliminar valores nulos en la tabla 'orders' para las columnas numericas
orders_clean = orders_clean.dropna(subset=cols_num).reset_index(drop=True)

# Eliminar valores nulos en la tabla 'orders' para las columnas categoricas
orders_clean = orders_clean.dropna(subset=cols_cat).reset_index(drop=True)

# Contar el pct de valores nulos en la tabla 'orders'
print(orders_clean.isna().mean().sort_values(ascending=False) * 100)

id_pedido             0.0
id_usuario            0.0
fecha_hora_pedido     0.0
pais                  0.0
dispositivo           0.0
fuente_referencia     0.0
nombre_producto       0.0
categoria_producto    0.0
cantidad              0.0
precio_unitario       0.0
monto_descuento       0.0
monto_total           0.0
dtype: float64


In [24]:
# Cantida de valores negativos en columnas numericas para table 'orders'
num_cols = ['cantidad', 'precio_unitario', 'monto_descuento', 'monto_total']

for col in num_cols:
    valores_negativos = len(orders_clean[orders_clean[col] < 0])
    print(f"La cantidad de valores negativos en la columna {col} es: {valores_negativos}")

La cantidad de valores negativos en la columna cantidad es: 4
La cantidad de valores negativos en la columna precio_unitario es: 0
La cantidad de valores negativos en la columna monto_descuento es: 0
La cantidad de valores negativos en la columna monto_total es: 4


In [25]:
# Marcar valores negativos como NaN
orders_clean.loc[orders_clean['cantidad'] <= 0, 'cantidad'] = np.nan

# Eliminar los valores negativos de la columna 'cantidad' de la tabla 'orders'
orders_clean = orders_clean.dropna(subset=['cantidad']).reset_index(drop=True)

# Verificar la cantidad de valores nulos para la columna 'cantidad'
print(orders_clean['cantidad'].isna().sum())

0


In [26]:
# Convertir la columna numerica 'cantidad' de float64 a int64
orders_clean['cantidad'] = orders_clean['cantidad'].fillna(0).astype('int')
print(f"El tipo de dato de la columna 'monto_descuento' ahora es {orders_clean['cantidad'].dtype}")


El tipo de dato de la columna 'monto_descuento' ahora es int64


In [27]:
# TABLA 'marketing'
# Como manejar los nulos en la columna categorica 'canal'
# Imputar como 'NA'

# Estandarizar la columna 'canal'
marketing_clean['canal'] = marketing_clean['canal'].str.strip().str.title()

#marketing_clean['id_campaña'].unique()

# Estandarizar la columna 'id_campaña'
marketing_clean['id_campaña'] = marketing_clean['id_campaña'].str.strip().str.title()

marketing_clean.head()

,fecha,pais,id_campaña,canal,gasto
0,2025-01-01,Mexico,Organic_Mexico,Organic,2446.25
1,2025-01-01,Mexico,Paid_Search_Mexico,Paid_Search,2704.34
2,2025-01-01,Mexico,Social_Mexico,Social,2045.01
3,2025-01-01,Colombia,Organic_Colombia,Organic,2597.21
4,2025-01-01,Colombia,Paid_Search_Colombia,Paid_Search,1771.40


In [28]:
marketing_clean.isna().mean().sort_values(ascending=False) * 100

canal         6.234568
fecha         0.000000
pais          0.000000
id_campaña    0.000000
gasto         0.000000
dtype: float64

In [29]:
marketing_clean[marketing_clean['canal'].isna()].head()

,fecha,pais,id_campaña,canal,gasto
98,2025-01-11,Argentina,Social_Argentina,NaN,849.70
99,2025-01-12,Mexico,Organic_Mexico,NaN,2033.56
100,2025-01-12,Mexico,Paid_Search_Mexico,NaN,1260.65
101,2025-01-12,Mexico,Social_Mexico,NaN,1660.90
102,2025-01-12,Colombia,Organic_Colombia,NaN,1819.27


In [30]:
# Fills missing 'canal' values using the prefix from 'id_campaña'
# Example: "Paid_Search_Mexico" -> "Paid_Search"
marketing_clean['canal'] = marketing_clean['canal'].fillna(
    marketing_clean['id_campaña'].str.rsplit('_', n=1).str[0]
)

# Verificar si hay nulos en la tabla de marketing
display(marketing_clean.isna().mean().sort_values(ascending=False) * 100)

fecha         0.0
pais          0.0
id_campaña    0.0
canal         0.0
gasto         0.0
dtype: float64

In [31]:
# Nada que procesar en la tabla de 'catalog'
catalog_clean = catalog.copy()

---
**📦 Exportación**: Una vez finalizada la limpieza, se exportan los datasets para utilizarlos en la última etapa del proyecto.

In [32]:
# exportar datasets
orders_clean.to_csv('orders_clean.csv', index=False)
catalog_clean.to_csv('catalog_clean.csv', index=False)
marketing_clean.to_csv('marketing_clean.csv', index=False)

---

## 🔹 Paso 2: Analizar si el negocio es rentable

### 2.1 Cálculo de KPIs principales

**🎯 Objetivo:** Calcular los indicadores clave del negocio para evaluar ingresos, costos y rentabilidad.

Se usan los 3 datasets (`orders`, `catalog`, `marketing_spend`):

**📊 Parte 1: Rentabilidad del negocio**
- ¿Cuál es el ingreso total (revenue)? 
- ¿Cuál es el costo total? 
- ¿Cuánto se ha invertido en marketing? 
- ¿El negocio es rentable? (calcular profit)  

---

**📈 Parte 2: Comportamiento de ventas**
- ¿Cuál es el ticket promedio por orden? 
- ¿Cuál es la cantidad promedio de productos por orden? 
- ¿Cuál es el producto más vendido?
- ¿Cuánto se ha gastado en marketing por canal? 

In [33]:
# Ingresos totales
ingreso_total = orders_clean['monto_total'].sum()
print(ingreso_total)

51941549.74000001


In [34]:
# Costo de marketing
costo_marketing = marketing_clean['gasto'].sum()
print(costo_marketing)

2871843.53


In [35]:
# Unir las tablas de 'orders_clean' y 'catalog_clean' para calcular el costo total
merged_orders_catalog = pd.merge(orders_clean, catalog_clean, on=['nombre_producto'], how='left')
merged_orders_catalog.head()

,id_pedido,id_usuario,fecha_hora_pedido,pais,dispositivo,fuente_referencia,nombre_producto,categoria_producto_x,cantidad,precio_unitario,monto_descuento,monto_total,categoria_producto_y,costo_unitario,proveedor
0,order_0,user_6993,2025-05-22,Argentina,Desktop,Organic,Jacket-Winter-M,Moda,2,332.69,0,665.37,Moda,189.31,Mcmillan-Rhodes
1,order_1,user_1329,2025-06-15,Mexico,Desktop,Paid_Search,Tablet-Standard-64GB,Electronica,1,176.86,5,171.86,Electrónica,25.21,Bowers LLC
2,order_2,user_3194,2025-05-02,Argentina,Desktop,Social,Blender-XL-Red,Hogar,2,102.99,10,195.99,Hogar,176.64,Long-Reid
3,order_3,user_4510,2025-06-09,Colombia,Mobile,Social,Tablet-Standard-64GB,Electronica,1,257.87,15,242.87,Electrónica,25.21,Bowers LLC
4,order_4,user_5044,2025-03-30,Argentina,Desktop,Paid_Search,Blender-XL-Red,Hogar,1,336.28,0,336.28,Hogar,176.64,Long-Reid


In [36]:
# Calcular el costo total
#costo_total = (merged_orders_catalog['costo_unitario'] * merged_orders_catalog['cantidad']).sum()
merged_orders_catalog['costo_total'] = merged_orders_catalog['costo_unitario'] * merged_orders_catalog['cantidad']
costo_total = merged_orders_catalog['costo_total'].sum()

print(costo_total)

43119535.88999999


In [37]:
# Forces Pandas to display standard numbers with 2 decimal places
pd.options.display.float_format = '{:.2f}'.format

# Agrupar por nombre de producto y sumar costos totales
costo_total_por_producto = merged_orders_catalog.groupby('nombre_producto')['costo_total'].sum().reset_index()

print(costo_total_por_producto)

        nombre_producto  costo_total
0        Blender-XL-Red   1107179.52
1       Jacket-Winter-M   1183376.81
2    Laptop-Gaming-16GB  40472371.92
3       Phone-Pro-128GB     41805.72
4     Sneakers-Urban-42    106082.44
5  Tablet-Standard-64GB    104571.08
6      Vacuum-Pro-Black    104148.40


In [38]:
# Calcular profit
profit = ingreso_total - (costo_marketing + costo_total)

print(profit)

5950170.320000015


In [39]:
# Calcular el ticket promedio
ticket_promedio = orders_clean.groupby('id_pedido')['monto_total'].sum().mean()

print(ticket_promedio)

2087.9346279696106


In [40]:
# Calcular la cantidad promedio de productos por orden
cantidad_prod_promedio = orders_clean.groupby('id_pedido')['cantidad'].sum().mean()

print(cantidad_prod_promedio)

7.132290871085742


In [41]:
# Calcular el producto mas vendido
producto_mas_vendido = orders_clean.groupby('nombre_producto')['cantidad'].sum().sort_values(ascending=False)

print(producto_mas_vendido)

nombre_producto
Laptop-Gaming-16GB      144194
Vacuum-Pro-Black          6274
Blender-XL-Red            6268
Jacket-Winter-M           6251
Sneakers-Urban-42         6164
Tablet-Standard-64GB      4148
Phone-Pro-128GB           4131
Name: cantidad, dtype: int64


In [42]:
# Calcular cuanto se ha gastado en marketing por canal
costo_mark_canal = marketing_clean.groupby('canal')['gasto'].sum().sort_values(ascending=False)

print(costo_mark_canal)

canal
Social        976818.37
Organic       972650.96
Paid_Search   922374.20
Name: gasto, dtype: float64


---

## 🔹 Paso 3: Entender dónde se pierden los usuarios (funnel de conversión)

**🎯 Objetivo:** Analizar el comportamiento de los usuarios para identificar en qué etapa del proceso se pierden.


⚙️**Conexión a la base de datos**:  
Se ejecuta la línea de configuración para conectar con la base de datos y aplicar consultas SQL en la tabla **events**.

---

**📊 Parte 1: Construcción del funnel**
- ¿Cuántos usuarios llegan a cada etapa del funnel?  
- Se calcula el número de usuarios únicos por `nombre_evento`  
- Se ordenan los eventos según el flujo del usuario  

---

**📉 Parte 2: Análisis de conversión**
- Se calcula la tasa de conversión entre cada paso del funnel  
- Se identifica en qué etapa se pierde la mayor cantidad de usuarios  
- ¿Cuál es la tasa de conversión final?
---

In [43]:

import pandas as pd
from sqlalchemy import create_engine

# ======================
# Conexión (NO modificar)
# ======================
db_config = {
    'user': 'practicum_student',
    'pwd': 'QnmDH8Sc2TQLvy2G3Vvh7',
    'host': 'yp-trainers-practicum.cluster-czs0gxyx2d8w.us-east-1.rds.amazonaws.com',
    'port': 5432,
    'db': 'data-analyst-production-db-en'
}

connection_string = 'postgresql://{}:{}@{}:{}/{}'.format(
    db_config['user'],
    db_config['pwd'],
    db_config['host'],
    db_config['port'],
    db_config['db']
)

engine = create_engine(connection_string, connect_args={'sslmode':'require'})


In [44]:
# Explorar tabla events
# =========================
query_events = '''
SELECT *
FROM events;
'''
events = pd.read_sql(query_events, con=engine)
events.head()



,id_usuario,id_sesion,nombre_evento,timestamp_evento,pais,dispositivo,fuente_referencia,categoria_producto
0,user_6772,6a97f2af-32ae-4186-8c92-04025be1a27b,first_visit,2025-05-17,Colombia,desktop,organic,Moda
1,user_5883,369b767c-1c33-4b2f-a652-c7c0ef92cfc9,add_to_cart,2025-02-23,Mexico,mobile,social,Hogar
2,user_5946,60039041-e78b-474c-87b3-c0b7e9c30708,add_payment_info,2025-05-15,Colombia,desktop,social,Electronica
3,user_827,18252a64-f389-4ef7-9e58-dadad4a3491e,purchase,2025-03-31,Mexico,mobile,social,Moda
4,user_2361,221b364e-cdc5-4668-b698-18d5ba849a67,first_visit,2025-01-22,Argentina,desktop,paid_search,Electronica


In [45]:
# PARTE 1: Totales del funnel
# ======================

query_totals = '''
SELECT 
    COUNT(id_usuario) AS cantidad_usuario,
    nombre_evento
FROM events
WHERE nombre_evento IN (
    'first_visit',
    'select_item',
    'add_to_cart',
    'begin_checkout',
    'add_payment_info'
)
GROUP BY nombre_evento
ORDER BY cantidad_usuario DESC;

'''

total_users = pd.read_sql(query_totals, con=engine)
total_users

,cantidad_usuario,nombre_evento
0,29957,first_visit
1,24157,add_to_cart
2,23887,select_item
3,17971,begin_checkout
4,12018,add_payment_info


In [46]:
# PARTE 2: Conversiones
# ======================

query_conversion = '''
WITH cte_first_visit AS (
    SELECT DISTINCT id_usuario
    FROM events
    WHERE nombre_evento = 'first_visit'
),
cte_select_item AS (
    SELECT DISTINCT id_usuario
    FROM events
    WHERE nombre_evento = 'select_item'
),
cte_add_to_cart AS (
    SELECT DISTINCT id_usuario
    FROM events
    WHERE nombre_evento = 'add_to_cart'
),
cte_begin_checkout AS (
    SELECT DISTINCT id_usuario
    FROM events
    WHERE nombre_evento = 'begin_checkout'
),
cte_add_payment_info AS (
    SELECT DISTINCT id_usuario
    FROM events
    WHERE nombre_evento = 'add_payment_info'
)
SELECT 
    (SELECT COUNT(*) FROM cte_first_visit) AS first_visit_count,
    (SELECT COUNT(*) FROM cte_select_item) AS select_item_count,
    (SELECT COUNT(*) FROM cte_add_to_cart) AS add_to_cart_count,
    (SELECT COUNT(*) FROM cte_begin_checkout) AS begin_checkout_count,
    (SELECT COUNT(*) FROM cte_add_payment_info) AS add_payment_info_count,

((SELECT COUNT(*) FROM cte_first_visit) - (SELECT COUNT(*) FROM cte_select_item)) * 100.0 
/ NULLIF((SELECT COUNT(*) FROM cte_first_visit), 0) AS drop_off_pct_select_item,

((SELECT COUNT(*) FROM cte_select_item) - (SELECT COUNT(*) FROM cte_add_to_cart)) * 100.0 
/ NULLIF((SELECT COUNT(*) FROM cte_select_item), 0) AS drop_off_pct_add_to_cart,

((SELECT COUNT(*) FROM cte_add_to_cart) - (SELECT COUNT(*) FROM cte_begin_checkout)) * 100.0 
/ NULLIF((SELECT COUNT(*) FROM cte_add_to_cart), 0) AS drop_off_pct_begin_checkout,

((SELECT COUNT(*) FROM cte_begin_checkout) - (SELECT COUNT(*) FROM cte_add_payment_info)) * 100.0 
/ NULLIF((SELECT COUNT(*) FROM cte_begin_checkout), 0) AS drop_off_pct_add_payment_info
'''

conversion = pd.read_sql(query_conversion, con=engine)
conversion

,first_visit_count,select_item_count,add_to_cart_count,begin_checkout_count,add_payment_info_count,drop_off_pct_select_item,drop_off_pct_add_to_cart,drop_off_pct_begin_checkout,drop_off_pct_add_payment_info
0,7796,7582,7634,7208,6250,2.74,-0.69,5.58,13.29


---

## 🔹 Paso 4: Evaluar si los usuarios regresan (retención por cohortes)

**🎯 Objetivo:** Analizar la retención de usuarios para entender si regresan después de registrarse.

**Tablas**

- `users` 
- `user_activity` 

---
1. Se identifica la cohorte de cada usuario según el **mes de registro**.


2. Se calcula la retención semanal: cuántos usuarios **se mantienen activos** en cada semana desde su registro.
   - `retenido_w1`: usuarios activos en la semana 1  
   - `retenido_w2`: usuarios activos en la semana 2  
   - `retenido_w3`: usuarios activos en la semana 3  


3. Se calcula el porcentaje de retención para cada semana, dividiendo los usuarios retenidos entre los clientes iniciales de la cohorte:  
   - `semana_1`: porcentaje de usuarios retenidos en la semana 1  
   - `semana_2`: porcentaje de usuarios retenidos en la semana 2  
   - `semana_3`: porcentaje de usuarios retenidos en la semana 3  

Se revisa que la columna de fecha esté en formato correcto (`DATE`).  
Se realiza la conversión usando: `CAST(fecha_registro AS DATE)`

In [47]:
# Explorar tabla users
# =========================
query_users = '''
SELECT *
FROM users;
'''
users = pd.read_sql(query_users, con=engine)
users.head(3)

,id_usuario,fecha_registro,país,dispositivo,tipo_plan
0,user_0,2025-01-29,Mexico,mobile,free
1,user_1,2025-01-07,Mexico,mobile,free
2,user_2,2025-03-12,Argentina,mobile,free


In [48]:
# Explorar tabla user_activity
# =========================

# NO hay usuarios en la tabla 'user_activity' con mas de 4 registros.

query_user_activity = '''

SELECT
    COUNT(*) AS usuarios_actividad_mayor_a_4
FROM (
    SELECT DISTINCT id_usuario,
    COUNT(id_usuario) AS conteo_actividad_usuario
    FROM user_activity
    GROUP BY id_usuario
    HAVING COUNT(id_usuario) > 4
    ORDER BY conteo_actividad_usuario DESC
) AS subquery;
'''
user_activity = pd.read_sql(query_user_activity, con=engine)
user_activity

,usuarios_actividad_mayor_a_4
0,0


In [49]:
# Comprobando si hay usuarios con mas de una fecha de registro en la tabla de 'users'

test_clientes_inicial = """
SELECT
    COUNT(*) AS usuarios_con_duplicado_fecha_registro
FROM (SELECT 
    MIN(fecha_registro) AS fecha_registro
    FROM users 
    GROUP BY id_usuario
    HAVING COUNT(fecha_registro) > 1
) AS subquery;
"""

# Ejecutar la consulta
test_clientes_inicial = pd.read_sql(test_clientes_inicial, con=engine)
test_clientes_inicial

,usuarios_con_duplicado_fecha_registro
0,0


In [50]:
# Consultar los tipos de datos de la tabla 'users'

tipo_datos_users = """
SELECT column_name, data_type
FROM information_schema.columns
WHERE table_name = 'users' AND table_schema = 'public';
"""
tipo_datos_users = pd.read_sql(tipo_datos_users, con=engine)
tipo_datos_users



,column_name,data_type
0,id_usuario,text
1,fecha_registro,text
2,país,text
3,dispositivo,text
4,tipo_plan,text


In [51]:
# Consultar los tipos de datos de la tabla 'user_activity'

tipo_datos_user_activity = """
SELECT column_name, data_type
FROM information_schema.columns
WHERE table_name = 'user_activity' AND table_schema = 'public';
"""
tipo_datos_user_activity = pd.read_sql(tipo_datos_user_activity, con=engine)
tipo_datos_user_activity

,column_name,data_type
0,id_usuario,text
1,fecha_actividad,text
2,dias_despues_registro,bigint
3,activo,bigint


In [52]:
# Verificando datos de la tabla 'user_activity'

cohorte_periodo = """
SELECT *
FROM user_activity;
"""

cohorte_periodo = pd.read_sql(cohorte_periodo, con=engine)
cohorte_periodo.head()


,id_usuario,fecha_actividad,dias_despues_registro,activo
0,user_0,2025-02-05,7,0
1,user_0,2025-02-12,14,1
2,user_0,2025-02-19,21,1
3,user_0,2025-02-26,28,0
4,user_1,2025-01-14,7,0


In [53]:
# Analizando la fecha de cohorte

cohorte_fecha = """

SELECT id_usuario,
    MIN(fecha_registro) AS cohort_fecha
FROM users
GROUP BY id_usuario;

"""

cohorte_fecha = pd.read_sql(cohorte_fecha, con=engine)
cohorte_fecha.head()

,id_usuario,cohort_fecha
0,user_3160,2025-01-31
1,user_5838,2025-01-13
2,user_6035,2025-05-12
3,user_7849,2025-05-02
4,user_1361,2025-03-14


In [54]:
# Retención por cohortes
# ======================

query_cohort_retention_final = '''
WITH cte_cohorte AS (
    SELECT 
        id_usuario,
        DATE_TRUNC('week', fecha_registro::date) AS cohort_semana
    FROM users
),
cte_retencion_mes AS (
    SELECT
        c.cohort_semana,
        COUNT(DISTINCT c.id_usuario) AS clientes_iniciales,
        COUNT(DISTINCT CASE WHEN ua.dias_despues_registro BETWEEN 7 AND 13 AND ua.activo = 1 THEN c.id_usuario END) AS retenido_w1,
        COUNT(DISTINCT CASE WHEN ua.dias_despues_registro BETWEEN 14 AND 20 AND ua.activo = 1 THEN c.id_usuario END) AS retenido_w2,
        COUNT(DISTINCT CASE WHEN ua.dias_despues_registro BETWEEN 21 AND 27 AND ua.activo = 1 THEN c.id_usuario END) AS retenido_w3
    FROM cte_cohorte AS c
    LEFT JOIN user_activity AS ua
        ON c.id_usuario = ua.id_usuario
    GROUP BY c.cohort_semana
)

SELECT
    TO_CHAR(cohort_semana, 'YYYY-MM-DD') AS cohort_semana,
    clientes_iniciales,
    ROUND((retenido_w1::numeric / NULLIF(clientes_iniciales, 0)) * 100, 2) AS semana_1,
    ROUND((retenido_w2::numeric / NULLIF(clientes_iniciales, 0)) * 100, 2) AS semana_2,
    ROUND((retenido_w3::numeric / NULLIF(clientes_iniciales, 0)) * 100, 2) AS semana_3
FROM  cte_retencion_mes
ORDER BY cohort_semana;
'''

# Ejecutar la consulta
cohorte_final = pd.read_sql(query_cohort_retention_final, con=engine)
cohorte_final.head()

,cohort_semana,clientes_iniciales,semana_1,semana_2,semana_3
0,2024-12-30,236,41.95,38.56,40.25
1,2025-01-06,351,41.31,44.73,42.17
2,2025-01-13,362,43.65,38.12,41.99
3,2025-01-20,394,44.16,39.59,39.85
4,2025-01-27,373,41.55,47.45,39.95


---

## 🔹 Paso 5: Validar si los cambios generan impacto (test estadístico)

🎯 **Objetivo:** Evaluar si la modificación en la UI del checkout impacta la **tasa de conversión de compra**.

---

1. **Analizar el dataset** `experiment_checkout_ui.csv` para identificar la métrica principal **conversion**.
   - La métrica **conversion** es 1 si el usuario completó la compra, 0 si no.    
2. **Plantear la hipótesis estadística**     
3. **Aplicar el test estadístico adecuado** 
4. **Interpretar el resultado**  

---
Hipótesis estadística
   - **H₀ (Hipótesis nula):** No hay ninguna diferencia en la conversion entre el grupo de "tratamiento" y el grupo de "control"
   - **H₁ (Hipótesis alternativa):** Si hay diferencia en la conversion entre el grupo de "tratamiento" y el grupo de "control"
   
- **Test estadístico:** z-test.
- **Nivel de significancia alpha:** 0.05.

In [55]:
# tu código aquí
exp_checkout = pd.read_csv('https://practicum-content.s3.amazonaws.com/datasets/experiment_checkout_ui.csv')
exp_checkout.head()

,id_usuario,variante,convirtio,dispositivo,pais,duracion_sesion,timestamp
0,exp_user_0,tratamiento,0,mobile,Argentina,114.41,2025-03-28
1,exp_user_1,tratamiento,0,desktop,Mexico,170.03,2025-01-15
2,exp_user_2,control,1,mobile,Colombia,140.21,2025-03-18
3,exp_user_3,tratamiento,0,mobile,Colombia,151.45,2025-06-03
4,exp_user_4,tratamiento,0,desktop,Mexico,299.96,2025-01-12


In [56]:
# Just some standard checking
test = exp_checkout['id_usuario'].unique().tolist()
test[:5]

['exp_user_0', 'exp_user_1', 'exp_user_2', 'exp_user_3', 'exp_user_4']

In [57]:
exp_checkout.shape

(10000, 7)

In [58]:
exp_checkout.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 7 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   id_usuario       10000 non-null  object 
 1   variante         10000 non-null  object 
 2   convirtio        10000 non-null  int64  
 3   dispositivo      10000 non-null  object 
 4   pais             10000 non-null  object 
 5   duracion_sesion  10000 non-null  float64
 6   timestamp        10000 non-null  object 
dtypes: float64(1), int64(1), object(5)
memory usage: 547.0+ KB


In [59]:
# Verificar si hay valores nulos en el dataset 'experiment_checkout_ui'
exp_checkout.isna().sum()

id_usuario         0
variante           0
convirtio          0
dispositivo        0
pais               0
duracion_sesion    0
timestamp          0
dtype: int64

In [60]:
# Estandarizar las columnas ['variante', 'dispositivo']
exp_checkout_clean = exp_checkout.copy()

def estandarizar_col_cat(df, col_cat):
    for col in col_cat:
        df[col] = df[col].str.strip().str.title()
    return df

exp_checkout_clean = estandarizar_col_cat(exp_checkout_clean, ['variante', 'dispositivo'])
exp_checkout_clean.head()


,id_usuario,variante,convirtio,dispositivo,pais,duracion_sesion,timestamp
0,exp_user_0,Tratamiento,0,Mobile,Argentina,114.41,2025-03-28
1,exp_user_1,Tratamiento,0,Desktop,Mexico,170.03,2025-01-15
2,exp_user_2,Control,1,Mobile,Colombia,140.21,2025-03-18
3,exp_user_3,Tratamiento,0,Mobile,Colombia,151.45,2025-06-03
4,exp_user_4,Tratamiento,0,Desktop,Mexico,299.96,2025-01-12


In [61]:
# Funcion para validar la cantidad de valores negativos en una columna.
def val_negativos(df, col_num1):
    resultado = {}

    for col in col_num1:
        valores_negativos = len(df[df[col] < 0])
        resultado[col] = valores_negativos

    return resultado

val_negativos(exp_checkout_clean, ['duracion_sesion'])


{'duracion_sesion': 0}

In [62]:
# Funcion para eliminar valores negativos de columnas numericas

def limpiar_valores_negativos(df, cols_num):
    for col in cols_num:
        df[col] = pd.to_numeric(df[col], errors='coerce')
        
        if (df[col] < 0).any():
            print(f"La columna '{col}' contiene valores negativos. Limpiando...")
        else:
            print(f"La columna '{col}' no contiene valores negativos.")
            
        df.loc[df[col] < 0, col] = np.nan
        
    return df

limpiar_valores_negativos(exp_checkout_clean, ['duracion_sesion']).head()

La columna 'duracion_sesion' no contiene valores negativos.


,id_usuario,variante,convirtio,dispositivo,pais,duracion_sesion,timestamp
0,exp_user_0,Tratamiento,0,Mobile,Argentina,114.41,2025-03-28
1,exp_user_1,Tratamiento,0,Desktop,Mexico,170.03,2025-01-15
2,exp_user_2,Control,1,Mobile,Colombia,140.21,2025-03-18
3,exp_user_3,Tratamiento,0,Mobile,Colombia,151.45,2025-06-03
4,exp_user_4,Tratamiento,0,Desktop,Mexico,299.96,2025-01-12


In [63]:
# Funcion para automatizar la estandarizacion de fechas

def estandarizar_fecha(df, date_col):
    for col in date_col:
        df[col] = pd.to_datetime(df[col], errors='coerce')

estandarizar_fecha(exp_checkout_clean, ['timestamp'])

exp_checkout_clean.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 7 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   id_usuario       10000 non-null  object        
 1   variante         10000 non-null  object        
 2   convirtio        10000 non-null  int64         
 3   dispositivo      10000 non-null  object        
 4   pais             10000 non-null  object        
 5   duracion_sesion  10000 non-null  float64       
 6   timestamp        10000 non-null  datetime64[ns]
dtypes: datetime64[ns](1), float64(1), int64(1), object(4)
memory usage: 547.0+ KB


In [64]:
# Sumar la cantidad de conversiones entre el grupo de control y tratamiento.
conversiones = exp_checkout_clean.groupby('variante')['convirtio'].sum()
conversiones

variante
Control        779
Tratamiento    820
Name: convirtio, dtype: int64

In [65]:
# Sumar la cantidad de observaciones entre el grupo de control y tratamiento.
observaciones = exp_checkout_clean.groupby('variante')['convirtio'].count()
observaciones

variante
Control        4965
Tratamiento    5035
Name: convirtio, dtype: int64

In [66]:
# Ordenar la cantidad de conversiones en una lista para usar en la prueba z-test
conversiones_ordenado = [conversiones['Control'], conversiones['Tratamiento']]
conversiones_ordenado

[779, 820]

In [67]:
# Ordenar la cantidad de observaciones en una lista para usar en la prueba z-test
observaciones_ordenado = [observaciones['Control'], observaciones['Tratamiento']]
observaciones_ordenado

[4965, 5035]

In [68]:
# Obtener tasa o porcentaje de conversion
tasa_control = conversiones_ordenado[0] / observaciones_ordenado[0]
tasa_tratamiento = conversiones_ordenado[1] / observaciones_ordenado[1]

print(f"Tasa de conversión del grupo de control: {tasa_control:.2%}")
print(f"Tasa de conversión del grupo de tratamiento: {tasa_tratamiento:.2%}")

# Interpretación de los resultados
if tasa_tratamiento > tasa_control:
    print("\nEl grupo de tratamiento tiene una tasa de conversión más alta que el grupo de control.")
else:
    print("\nEl grupo de tratamiento no tiene una tasa de conversión más alta que el grupo de control.")  

Tasa de conversión del grupo de control: 15.69%
Tasa de conversión del grupo de tratamiento: 16.29%

El grupo de tratamiento tiene una tasa de conversión más alta que el grupo de control.


In [69]:
# Aplicar la prueba z-test
from statsmodels.stats.proportion import proportions_ztest

z_stat, p_value = proportions_ztest(conversiones_ordenado, observaciones_ordenado)

print(f"Estadistico Z: {z_stat}")
print(f"P-value: {p_value}")


# Interpretación de resultados
if p_value < 0.05:
    print("\nHay evidencia suficiente para rechazar la hipótesis nula. Las conversiones entre las variantes son significativamente diferentes.")
else:
    print("\nNo hay evidencia suficiente para rechazar la hipótesis nula. No se encontraron diferencias significativas en las conversiones entre las variantes.")

Estadistico Z: -0.8132782986429474
P-value: 0.41605851639119995

No hay evidencia suficiente para rechazar la hipótesis nula. No se encontraron diferencias significativas en las conversiones entre las variantes.


---

## 🔹 Paso 6: Comunicar los resultados (Dashboard en BI)

🎯 **Objetivo**:  
Crear un dashboard que muestre de manera clara y visual los resultados del análisis de ventas, costos, marketing y conversión. 

Se usarán los CSVs limpios del Paso 1:

- `orders_clean.csv`  
- `catalog_clean.csv`  
- `marketing_clean.csv`

---

1️⃣ Preparación de los datos
1. Cargar los CSVs en Power BI o Tableau.
2. Revisar relaciones:
   - `orders.nombre_producto` → `catalog.nombre_producto`
   - `orders.fecha_pedido` → tabla de fechas (crear calendario para análisis temporal)
   - `orders.fecha_pedido` → `dim_fecha.date`
3. Crear columnas calculadas necesarias
4. Crear tabla de fechas para poder calcular comparaciones YTD, YoY o períodos anteriores (`Previous Year`, `Previous Month`).

---

2️⃣ Dashboard 1: Overview Ejecutivo
**KPIs principales a mostrar:**
- Revenue total
- Profit total
- Gasto total en marketing
- Ticket promedio
- Cantidad promedio de productos por orden

**Visualizaciones sugeridas:**
- Tarjetas KPI para revenue, profit y gasto marketing
- Gráfico de líneas: evolución mensual de revenue o profit
- Gráfico de líneas YTD
- Gráfico de barras: revenue y profit por producto o categoría

---

 3️⃣ Dashboard 2: Detalle / Drill-through  
**Objetivo:** Permitir explorar los datos desde el KPI general hasta cada orden o producto.

**Visualizaciones sugeridas:**
- Tabla detallada de órdenes con:
  - producto, cantidad, revenue, cost, profit
  - color condicional (profit negativo en rojo, positivo en verde)
- Gráfico de barras por producto con medida `cantidad vendida`
- Drill-through: seleccionar un producto y ver todos los pedidos relacionados
- Filtros por fecha, categoría de producto, etc

---

## 🚀 Entrega Final

Comparte el acceso a tu Dashboard para revisión.   
Puedes entregar el Dashboard utilizando **Power BI o Tableau**.

Incluye **uno de los siguientes**:

- 🔗 Link público del dashboard publicado en **Power BI Service o Tableau Public / Tableau Cloud**
- 🔗 Link de **Google Drive o OneDrive** con el archivo del proyecto (`.pbix`) y los 3 csvs limpios.


### 📎 Enlace del Dashboard

In [70]:
# (Pega aquí tu link)
# link de power bi o tableau
# link de one drive / google drive